# exp-036 — Qwen3.5-9B pool-only QLoRA fine-tune (replication of exp-026, one change: the base model)

Run this top to bottom (**Runtime → Run all**). Every step writes to Google Drive as it goes and
**skips itself when its output is already there**, so after a disconnect you reconnect and press
Run all again: it resumes at the first unfinished step. (Training itself cannot resume mid-run: an
interrupted training restarts from step 0 with the same seed; finished epochs are on Drive as
checkpoints for the record.)

**GPU:** pick **L4** (default) or **A100** in *Runtime → Change runtime type*. The notebook refuses
anything with < 22 GiB or without bf16 (T4, V100). Requirement and estimates are in
`experiments/exp-036/run_config.json` (`gpu`, `wall_clock_estimate`) and printed below.

**Before running:** the branch `feature/50-qwen-pool-finetune` must be pushed to GitHub, and
`MyDrive/thesis-backup.tar.gz` must exist.

**Commit pinning:** the first run pins the commit it cloned (`MyDrive/thesis-exp-036/pinned_commit.txt`);
every later run checks out exactly that commit, and training and both evaluations must carry it.
To pin a specific commit instead, set `COMMIT` in the first cell before the first run.

What it does: mount Drive → clone and pin the commit → check the GPU → build a Python 3.13.9 venv
with the repo's pinned versions → build `../thesis-qav2-data` from the backup → verify input hashes →
download Qwen3.5-9B at the pinned revision → **train** (4-bit QLoRA, 250 steps) → **evaluate at
4-bit (PRIMARY, precision-matched to exp-027)** → **evaluate at fp16** (exp-026's protocol;
secondary) → collect the files to copy back (after each evaluation).

In [ ]:
# ---- Parameters (the only cell you might edit) ----
REPO_URL = "https://github.com/AhmadiJahid/Thesis---QAv2.git"
BRANCH = "feature/50-qwen-pool-finetune"
COMMIT = ""   # "" = pin the branch head on the first run (recorded on Drive); or a full 40-char SHA
BACKUP_TAR = "/content/drive/MyDrive/thesis-backup.tar.gz"
DRIVE_OUT = "/content/drive/MyDrive/thesis-exp-036"   # everything that must survive a disconnect

PARENT = "/content/thesis"                 # <parent>/Thesis---QAv2 and <parent>/thesis-qav2-data
REPO = f"{PARENT}/Thesis---QAv2"
DATA = f"{PARENT}/thesis-qav2-data"
VENV = "/content/venv313"
PY = f"{VENV}/bin/python"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, subprocess, shutil, time, glob, tarfile, datetime
os.makedirs(f"{DRIVE_OUT}/logs", exist_ok=True)
os.makedirs(PARENT, exist_ok=True)

def sh(cmd, log=None, env=None, cwd=None):
    # Run a shell command, stream its output here and (optionally) append it to a Drive log. Raises on rc != 0.
    print(f"$ {cmd}", flush=True)
    full_env = {**os.environ, **(env or {})}
    with subprocess.Popen(cmd, shell=True, cwd=cwd, env=full_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash") as p:
        fh = open(log, "a") if log else None
        if fh: fh.write(f"\n### {datetime.datetime.now().isoformat()} $ {cmd}\n")
        for line in p.stdout:
            print(line, end="", flush=True)
            if fh: fh.write(line); fh.flush()
        rc = p.wait()
        if fh: fh.write(f"### rc={rc}\n"); fh.close()
    if rc != 0:
        raise RuntimeError(f"command failed (rc={rc}): {cmd}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def done(marker):  return os.path.exists(f"{DRIVE_OUT}/{marker}.done")
def mark(marker):  open(f"{DRIVE_OUT}/{marker}.done", "w").write(datetime.datetime.now().isoformat())

## 1. Clone the repo and pin the commit

In [ ]:
import re
if not os.path.isdir(f"{REPO}/.git"):
    sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin {BRANCH}")
PIN_FILE = f"{DRIVE_OUT}/pinned_commit.txt"
pinned = open(PIN_FILE).read().strip() if os.path.exists(PIN_FILE) else None
if COMMIT and pinned and COMMIT != pinned:
    raise SystemExit(f"COMMIT={COMMIT} differs from the commit this run already pinned ({pinned}) in {PIN_FILE}. "
                     "Finished steps on Drive were produced at the pinned commit; use it, or start a fresh DRIVE_OUT.")
SHA = COMMIT or pinned or subprocess.check_output(["git", "-C", REPO, "rev-parse", f"origin/{BRANCH}"], text=True).strip()
assert re.fullmatch(r"[0-9a-f]{40}", SHA), f"COMMIT must be a full 40-hex SHA, got {SHA!r}"
sh(f"git -C {REPO} checkout --detach {SHA}")
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head == SHA, f"checked out {head}, expected {SHA}"
if not pinned:
    open(PIN_FILE, "w").write(SHA + "\n")
print("pinned commit:", SHA)
CFG = json.load(open(f"{REPO}/experiments/exp-036/run_config.json"))

## 2. GPU check (aborts on an insufficient GPU)

In [ ]:
g = CFG["gpu"]
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                             "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
name, mem_mib, cc = [x.strip() for x in q.split(",")]
mem_gib = float(mem_mib) / 1024
cc_t = tuple(int(x) for x in cc.split("."))
print(f"GPU: {name} | {mem_gib:.1f} GiB | compute capability {cc}")
print("Requirement:", g["requirement_text"])
print("Estimated peaks (NOT measured):", "train", g["estimate_train_peak_gib"], "| eval fp16", g["estimate_eval_fp16_peak_gib"],
      "| eval 4bit", g["estimate_eval_4bit_peak_gib"])
if mem_gib < g["min_total_vram_gib"] or cc_t < tuple(g["min_compute_capability"]):
    raise SystemExit(f"ABORT: {name} ({mem_gib:.1f} GiB, sm_{cc}) does not meet the requirement "
                     f"(>= {g['min_total_vram_gib']} GiB and compute capability >= "
                     f"{'.'.join(map(str, g['min_compute_capability']))}). "
                     "Switch to an L4 or A100 runtime (Runtime -> Change runtime type) and run again.")
kind = "A100" if "A100" in name else ("L4" if "L4" in name else None)
est = CFG["wall_clock_estimate"].get(kind) if kind else None
if est:
    lo, hi = est["total_h_incl_setup"]
    rate = CFG["colab_compute_units"]["per_hour_approx"][kind]
    print(f"ESTIMATE (not measured) for {kind}: train {est['train_min']} min, eval fp16 {est['eval_fp16_min']} min, "
          f"eval 4bit {est['eval_4bit_min']} min; total {lo}-{hi} h incl. setup")
    print(f"ESTIMATED compute units: {lo*rate:.0f}-{hi*rate:.0f} CU at ~{rate} CU/h (UNVERIFIED rate - "
          "check Colab's Resources panel for the real one)")
else:
    print(f"No estimate for {name}; it passed the VRAM/bf16 gate, so the run can proceed.")
open(f"{DRIVE_OUT}/logs/gpu.txt", "a").write(f"{datetime.datetime.now().isoformat()} {q}\n")

## 3. Python 3.13.9 venv with the repo's pinned versions
exp-026 ran Python 3.13.9 + torch 2.10.0+cu128 (docs/compute.md). `uv` installs that exact interpreter, so Colab's own Python version does not matter. flash-linear-attention / causal-conv1d are deliberately **not** installed (exp-027 ran Qwen3.5 on the torch fallback).

In [ ]:
if not os.path.exists(f"{VENV}/.exp036_ok"):
    sh("pip -q install uv")
    sh(f"uv venv --python 3.13.9 {VENV}")
    sh(f"uv pip install --python {PY} torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128")
    sh(f"uv pip install --python {PY} -r {REPO}/requirements.txt")
    open(f"{VENV}/.exp036_ok", "w").write("ok")
VERS = f"{DRIVE_OUT}/logs/environment.json"
sh(f'''{PY} - <<'EOF' | tee {VERS}
import json, platform, sys, torch, transformers, peft, trl, bitsandbytes, accelerate, datasets, huggingface_hub
print(json.dumps({{"python": sys.version.split()[0], "platform": platform.platform(),
  "torch": torch.__version__, "cuda_runtime": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
  "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
  "transformers": transformers.__version__, "peft": peft.__version__, "trl": trl.__version__,
  "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__,
  "datasets": datasets.__version__, "huggingface_hub": huggingface_hub.__version__}}, indent=2))
EOF''')
env_rec = json.load(open(VERS))
want = {"python": "3.13.9", "torch": "2.10.0", "transformers": "5.5.4", "peft": "0.20.0", "trl": "1.10.0",
        "bitsandbytes": "0.49.2", "accelerate": "1.12.0", "datasets": "4.8.4",
        "huggingface_hub": "1.11.0"}
off = {k: (env_rec[k], v) for k, v in want.items() if not str(env_rec[k]).startswith(v)}
assert env_rec["cuda_available"] and env_rec["bf16_supported"], "CUDA/bf16 not available inside the venv"
assert not off, f"version drift from requirements.txt / exp-026: {off}"
print("versions match exp-026's environment")

## 4. Data: build `../thesis-qav2-data` from the Drive backup (docs/compute.md layout), place the two pinned artifacts, verify hashes
Whether to extract is decided from what is on this VM's disk (a marker under `DATA` plus the hash checks), not from Drive: after a VM reset the disk is empty and extraction runs again.

In [ ]:
LOCAL_TAR = "/content/thesis-backup.tar.gz"
RUNS = f"{REPO}/runs"
inp = CFG["inputs"]
pinned_runs = [inp["pool"]["path_under_runs"], inp["retrieval"]["path_under_runs"]]
DATA_MARKER = f"{DATA}/.exp036_data_ok"   # ephemeral, like DATA itself

def route(name):
    # Map a backup member to its destination, or None to skip. Layout: docs/compute.md 'Dataset location'.
    # Refuses path traversal: no '..' part, no absolute or drive-anchored name, and the resolved
    # destination must stay under DATA or RUNS.
    if name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", name) or "\\" in name:
        raise ValueError(f"refusing archive member with an absolute/drive-anchored/backslash name: {name!r}")
    parts = [p for p in name.split("/") if p not in ("", ".")]
    if ".." in parts:
        raise ValueError(f"refusing archive member with a '..' component: {name!r}")
    top = [i for i, p in enumerate(parts) if p in ("Thesis---QA", "Thesis---QAv2")]
    if not top:
        return None
    n = "/".join(parts[top[0]:])   # tolerate a leading folder in the archive
    dst = None
    for pre, base in (("Thesis---QA/MusiQue/Data/emb_cache/", f"{DATA}/cache/emb_cache/"),
                      ("Thesis---QA/MusiQue/Data/", f"{DATA}/musique/"),
                      ("Thesis---QA/Pool/", f"{DATA}/metaqa/pool/"),
                      ("Thesis---QA/Data/", f"{DATA}/metaqa/")):
        if n.startswith(pre) and len(n) > len(pre):
            dst = base + n[len(pre):]
            break
    if dst is None:
        for rel in pinned_runs:
            if n == f"Thesis---QAv2/runs/{rel}":
                dst = f"{RUNS}/{rel}"
    if dst is None:
        return None
    real = os.path.realpath(dst)
    if not any(real.startswith(os.path.realpath(root) + os.sep) for root in (DATA, RUNS)):
        raise ValueError(f"refusing archive member that resolves outside DATA/RUNS: {name!r} -> {real}")
    return dst

def inputs_ok():
    for key in ("pool", "retrieval"):
        p = f"{RUNS}/{inp[key]['path_under_runs']}"
        if not os.path.exists(p) or sha256(p) != inp[key]["sha256"]:
            return False
    return os.path.exists(DATA_MARKER)

if not inputs_ok():
    if not os.path.exists(LOCAL_TAR):
        sh(f"cp '{BACKUP_TAR}' {LOCAL_TAR}")
    n_written = 0
    with tarfile.open(LOCAL_TAR, "r:gz") as tf:
        for m in tf:
            dst = route(m.name)
            if dst is None or not (m.isfile() or m.isdir()):
                continue   # links, devices and unmapped members are never written
            if m.isdir():
                os.makedirs(dst, exist_ok=True); continue
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with tf.extractfile(m) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
            n_written += 1
    print("files written:", n_written)
    assert n_written > 0, "nothing matched Thesis---QA/{Data,MusiQue,Pool} or the two pinned runs/ files - check the archive layout (tar -tzf | head)"
    # reference/musique_official: not in git, fetched from the pinned upstream commit and hash-checked
    ref = inp["musique_official_reference"]
    os.makedirs(f"{DATA}/reference/musique_official", exist_ok=True)
    for fname, spec in ref["files"].items():
        dst = f"{DATA}/reference/musique_official/{fname}"
        url = f"https://raw.githubusercontent.com/StonyBrookNLP/musique/{ref['commit']}/{spec['upstream_path']}"
        sh(f"curl -sSfL '{url}' -o '{dst}'")
        assert sha256(dst) == spec["sha256"], f"{fname}: sha256 mismatch vs the archived copy"
    open(DATA_MARKER, "w").write(datetime.datetime.now().isoformat())

# Gates: the two pinned inputs are byte-identical to exp-026's (ADR 0035 amendment - never rebuilt)
for key in ("pool", "retrieval"):
    p = f"{RUNS}/{inp[key]['path_under_runs']}"
    assert os.path.exists(p), f"missing {p} - is it in the backup under Thesis---QAv2/runs/?"
    h = sha256(p); n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{key}: {h} rows={n}")
    assert h == inp[key]["sha256"] and n == inp[key]["rows"], f"{key}: hash/row mismatch vs exp-026"
# Every path the run reads resolves under the data root
paths = json.load(open(f"{REPO}/configs/paths.json"))
for hop in (2, 3, 4):
    f = f"{DATA}/" + paths["datasets"]["musique_eval_questions_template"].format(hop=hop)
    n = sum(1 for _ in open(f, encoding="utf-8")); print(f, n); assert n == 200, f
assert os.path.exists(f"{DATA}/" + paths["datasets"]["musique_dev_gold_clean"]), "gold dev file missing"
for d in ("metaqa", "metaqa/pool", "musique", "cache/emb_cache", "reference/musique_official"):
    print(f"{d}: {'ok' if os.path.isdir(f'{DATA}/{d}') else 'MISSING'}")

## 5. Base model at the pinned revision

In [ ]:
M = CFG["model"]
sh(f'''{PY} - <<'EOF'
from huggingface_hub import snapshot_download
p = snapshot_download("{M['model_id']}", revision="main")
print("snapshot:", p)
assert p.rstrip("/").endswith("{M['hf_revision']}"), "Qwen/Qwen3.5-9B main moved off the pinned revision - stop and tell the lead"
EOF''', log=f"{DRIVE_OUT}/logs/download.log")
HF_ENV = {"HF_HUB_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false"}

## 6. Outputs go straight to Drive
`runs/exp-036` in the clone is a symlink to Drive, so every checkpoint, adapter, prediction file and evaluation lands on Drive as it is written.

In [ ]:
os.makedirs(f"{DRIVE_OUT}/runs/exp-036", exist_ok=True)
link = f"{RUNS}/exp-036"
if not os.path.islink(link):
    if os.path.exists(link): shutil.rmtree(link)
    os.symlink(f"{DRIVE_OUT}/runs/exp-036", link)
print(link, "->", os.readlink(link))

class GpuSampler:
    # nvidia-smi memory.used every 5 s to a Drive file (exp-026 recorded the same kind of number)
    def __init__(self, tag): self.path = f"{DRIVE_OUT}/logs/vram_{tag}.csv"
    def __enter__(self):
        self.p = subprocess.Popen(f"nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -lms 5000 >> {self.path}",
                                  shell=True); return self
    def __exit__(self, *a):
        self.p.terminate()
        vals = [int(x) for x in open(self.path).read().split() if x.strip().isdigit()]
        print(f"peak nvidia-smi memory.used ({self.path}): {max(vals) if vals else 'n/a'} MiB")

def completed_train_runs():
    out = []
    for d in sorted(glob.glob(f"{RUNS}/exp-036/train/*/")):
        m = f"{d}metrics.json"
        if os.path.exists(m) and os.path.exists(f"{d}adapter/adapter_model.safetensors"):
            if json.load(open(m)).get("dry_run") is False:
                out.append(d.rstrip("/"))
    return out

## 7. Train (4-bit QLoRA, arm `pool_2000_imbalanced`, seed 42 — exp-026's command plus `--model qwen3_5_9b`)

In [ ]:
# Fail fast on a dirty tree: the training gate requires git.dirty == False in the run snapshot.
dirty = subprocess.check_output(["git", "-C", REPO, "status", "--porcelain"], text=True).strip()
assert not dirty, f"working tree is not clean (the run snapshot would record dirty=True): {dirty!r}"
if not completed_train_runs():
    partial = glob.glob(f"{RUNS}/exp-036/train/*/")
    if partial: print("NOTE: incomplete training dirs from an interrupted attempt (kept, ignored):", partial)
    with GpuSampler("train"):
        sh(f"{PY} " + " ".join(CFG["train"]["argv"]), cwd=REPO, env=HF_ENV, log=f"{DRIVE_OUT}/logs/train.log")
runs_done = completed_train_runs()
assert len(runs_done) == 1, f"expected exactly one completed training run, found {runs_done}"
TRAIN_DIR = runs_done[0]
tm = json.load(open(f"{TRAIN_DIR}/metrics.json"))
gate = CFG["train"]["gate"]
checks = {
    "training_source_sha256": tm["training_source_sha256"] == gate["training_source_sha256"],
    "num_examples": tm["dataset"]["num_examples"] == gate["num_examples"],
    "hop_counts": tm["dataset"]["hop_counts"] == gate["hop_counts"],
    "eval_overlap": tm["eval_overlap"]["overlap_count"] == gate["eval_overlap_count"],
    "quantization": tm["quantization"] == gate["quantization"],
    "model_id": tm["model_id"] == gate["model_id"],
    "global_step": tm["train_result"]["global_step"] == gate["global_step"],
    "ceiling_asserted": bool(tm["base_model_size"]["ceiling_asserted"]) and bool(tm["lora_model_size"]["ceiling_asserted"]),
    "commit": tm["git"]["commit"] == SHA,
    "clean_tree": tm["git"]["dirty"] is False,
}
print(json.dumps(checks, indent=1)); assert all(checks.values()), "TRAIN GATE FAILED - do not evaluate; tell the lead"
print("train wall clock s:", tm["wall_clock_seconds"], "| peak GPU GiB (max_memory_allocated):", tm["peak_gpu_memory_gib"])
print("trainable:", tm["trainable_parameters"]["trainable_parameters"], "/", tm["trainable_parameters"]["total_parameters"])
print("train loss:", tm["train_result"]["training_loss"])
ADAPTER = os.path.relpath(f"{TRAIN_DIR}/adapter", REPO)
TRAIN_COMMIT = tm["git"]["commit"]
print("training commit:", TRAIN_COMMIT)
print("adapter:", ADAPTER)

## 8. Evaluation helpers (gates, output health, collection)

In [ ]:
def out_root(tag):
    a = CFG["eval"][tag]["argv"]
    return f"{REPO}/" + a[a.index("--output-root") + 1]

def finished_eval(tag):
    # A decomposer run counts as finished only when its metrics.json exists (written last).
    return sorted(os.path.dirname(p) for p in glob.glob(f"{out_root(tag)}/*/metrics.json"))

def output_health(dec_dir, gate):
    rows = json.load(open(f"{dec_dir}/results.json"))
    marker = gate["degenerate_marker"]
    empty = sum(1 for r in rows if not str(r.get("decomposition", "")).strip())
    degenerate = sum(1 for r in rows if marker in str(r.get("decomposition_raw", "")))
    return {"rows": len(rows), "empty": empty, "degenerate": degenerate,
            "at_max_new_tokens": sum(1 for r in rows if r.get("hit_max_new_tokens"))}

def run_eval(tag):
    spec = CFG["eval"][tag]
    if not finished_eval(tag):
        argv = [a.replace("{adapter}", ADAPTER) for a in spec["argv"]]
        with GpuSampler("eval_" + tag):
            sh(f"{PY} " + " ".join(argv), cwd=REPO, env=HF_ENV, log=f"{DRIVE_OUT}/logs/eval_{tag}.log")
    done_dirs = finished_eval(tag)
    assert len(done_dirs) == 1, f"expected one finished decomposer run under {out_root(tag)}, found {done_dirs}"
    dec_dir = done_dirs[0]
    dm = json.load(open(f"{dec_dir}/metrics.json")); dc = json.load(open(f"{dec_dir}/config.json"))
    eg = CFG["eval"]["gate"]; oh = eg["output_health"]
    health = output_health(dec_dir, oh)
    json.dump(health, open(f"{DRIVE_OUT}/logs/output_health_{tag}.json", "w"), indent=1)
    checks = {
        "rows": dm["rows_loaded_total"] == eg["rows"],
        "rows_per_hop": {str(k): v for k, v in dm["rows_loaded_per_hop"].items()} == eg["rows_per_hop"],
        "retrieval_sha256": dc["retrieval"]["input_sha256"] == eg["retrieval_input_sha256"],
        "adapter_prompt_parity": dm["adapter_prompt_parity"]["checked"] and len(dm["adapter_prompt_parity"]["mismatches"]) == eg["adapter_prompt_parity_mismatches"],
        "no_few_shot": dm["no_few_shot"] is True,
        "quantization": dc["quantization"] == spec["argv"][spec["argv"].index("--quantization") + 1],
        "anchors": dm["anchor_prompts_logged"] == eg["anchor_prompts_expected"],
        "ceiling_asserted": bool(dm["model_size"]["ceiling_asserted"]),
        "same_commit_as_training": dc["git"]["commit"] == TRAIN_COMMIT == SHA,
        "empty_rows_within_threshold": health["empty"] <= oh["max_rows_with_empty_decomposition"]
                                        and dm["rows_with_empty_decomposition"] <= oh["max_rows_with_empty_decomposition"],
        "not_all_empty": health["empty"] < health["rows"],
        "no_degenerate_rows": health["degenerate"] <= oh["max_rows_degenerate"],
    }
    print(f"[{tag}] gate:", json.dumps(checks, indent=1))
    print(f"[{tag}] output health:", health)
    if not all(checks.values()):
        json.dump(checks, open(f"{DRIVE_OUT}/logs/GATE_FAILED_eval_{tag}.json", "w"), indent=1)
        raise SystemExit(f"EVAL GATE FAILED ({tag}) - recorded in logs/GATE_FAILED_eval_{tag}.json; tell the lead. "
                         "Do not re-run with changed settings (a changed run is a new experiment).")
    run_dir = spec["evaluator_run_dir"]
    if not os.path.exists(f"{REPO}/{run_dir}/eval_metrics.json"):
        res = os.path.relpath(f"{dec_dir}/results.json", REPO)
        ev = [a.replace("{results}", res).replace("{run_dir}", run_dir) for a in CFG["eval"]["evaluator_argv"]]
        sh(f"{PY} " + " ".join(ev), cwd=REPO, log=f"{DRIVE_OUT}/logs/evaluator_{tag}.log")
    em = json.load(open(f"{REPO}/{run_dir}/eval_metrics.json"))
    card = {k: em[k] for k in ("break_exact_match_rate", "sari_macro", "ged_macro", "chain_validity_macro",
                              "hop_count_exact_match_rate", "under_decomposition_rate", "over_decomposition_rate")}
    print(f"REPORT CARD exp-036 {tag} (n={em['total_evaluated']}):", json.dumps(card, indent=1))
    return dec_dir

def rd(p): return json.load(open(p))

def collect():
    # Bundle what goes back into the repo; safe to call after each evaluation.
    EXP = f"{DRIVE_OUT}/experiments_exp-036"; os.makedirs(EXP, exist_ok=True)
    snap = {"_note": "exp-036 config snapshots, copied verbatim from the run directories (written by the notebook; nothing here is a result).",
            "repo_commit": SHA, "environment": rd(f"{DRIVE_OUT}/logs/environment.json"),
            "training_run_config": rd(f"{TRAIN_DIR}/config.json"), "run_plan": CFG}
    met = {"_note": "exp-036 measured outputs, copied verbatim from the run directories by the notebook. No number here is typed by hand.",
           "repo_commit": SHA, "training_metrics": rd(f"{TRAIN_DIR}/metrics.json")}
    for tag in ("4bit", "fp16"):
        dirs = finished_eval(tag)
        rdir = f"{REPO}/" + CFG["eval"][tag]["evaluator_run_dir"]
        if dirs:
            snap[f"decomposer_run_config_{tag}"] = rd(f"{dirs[0]}/config.json")
            met[f"decomposer_metrics_{tag}"] = rd(f"{dirs[0]}/metrics.json")
        if os.path.exists(f"{rdir}/eval_metrics.json"):
            snap[f"evaluator_config_{tag}"] = rd(f"{rdir}/eval_config.json")
            met[f"eval_metrics_{tag}"] = rd(f"{rdir}/eval_metrics.json")
        for kind in ("output_health", "GATE_FAILED_eval"):
            f = f"{DRIVE_OUT}/logs/{kind}_{tag}.json"
            if os.path.exists(f): met[f"{kind}_{tag}"] = rd(f)
    vram = {}
    for f in glob.glob(f"{DRIVE_OUT}/logs/vram_*.csv"):
        vals = [int(x) for x in open(f).read().split() if x.strip().isdigit()]
        vram[os.path.basename(f)] = max(vals) if vals else None
    met["peak_nvidia_smi_memory_used_mib"] = vram
    met["gpu"] = open(f"{DRIVE_OUT}/logs/gpu.txt").read().strip().splitlines()
    json.dump(snap, open(f"{EXP}/config.json", "w"), indent=2, default=str)
    json.dump(met, open(f"{EXP}/metrics.json", "w"), indent=2, default=str)
    shutil.copy(f"{DRIVE_OUT}/logs/environment.json", f"{EXP}/environment.json")
    ZIP = f"{DRIVE_OUT}/exp-036_copyback.zip"
    sh(f"cd {DRIVE_OUT} && rm -f {ZIP} && zip -qr {ZIP} experiments_exp-036 logs pinned_commit.txt runs/exp-036 -x 'runs/exp-036/train/*/checkpoints/*'")
    print("\n" + "=" * 78)
    print("Bundle on Google Drive:", ZIP)
    print("Copy back on the laptop (repo = C:/Users/suuser/thesis/Thesis---QAv2):")
    print("  1. unzip exp-036_copyback.zip into a temp folder")
    print("  2. experiments_exp-036/config.json       -> experiments/exp-036/config.json")
    print("     experiments_exp-036/metrics.json      -> experiments/exp-036/metrics.json")
    print("     experiments_exp-036/environment.json  -> experiments/exp-036/environment.json")
    print("  3. runs/exp-036/  (whole folder)          -> runs/exp-036/   (gitignored; holds the adapter,")
    print("     results.json and eval_per_item.json that compare.py reads)")
    print("  4. logs/ and pinned_commit.txt            -> runs/exp-036/colab_logs/")
    print("Then follow experiments/exp-036/README.md 'After the run'.")
    for p in sorted(glob.glob(f"{DRIVE_OUT}/runs/exp-036/**/eval_per_item.json", recursive=True)):
        print("  per-item:", p, sha256(p))

## 9. Evaluate at 4-bit — PRIMARY (mandatory)
The same adapter on a 4-bit base: precision-matched to exp-027 (Qwen3.5-9B prompting, 4-bit), so pair (b) — the fine-tuning gain on Qwen — moves only adapter + prompt. Then the evaluator, then the bundle is written (so the primary result is safe even if the fp16 step fails).

In [ ]:
run_eval("4bit")
collect()

## 10. Evaluate at fp16 — exp-026's protocol (secondary)
fp16 base + adapter, exactly exp-026's evaluation. **On an L4 this is the tightest step (~18–19.5 GiB, estimate).** If it runs out of memory, everything so far is on Drive: switch to an A100 runtime and Run all again — it resumes here. If its gate fails (e.g. fp16 overflow), the primary result stands; run `compare.py --fp16-failed-gate` later.

In [ ]:
run_eval("fp16")
collect()